In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
#load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

In [ ]:
#load finetuned model
finetuned_model_path = "./final_model"
ft_model = PeftModel.from_pretrained(base_model, finetuned_model_path)

ft_model.eval()

In [ ]:
prompt = "Generate a medium difficulty dungeon with 6 rooms"

In [ ]:
messages = [{"role": "user", "content":prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

In [ ]:
#device management
device = next(ft_model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

In [ ]:
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=16384,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=None,
        repetition_penalty=1.0
    )

In [ ]:
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
generated_content = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(generated_content)

In [ ]:
import re
import json

def extract_json_from_text(text):
    """Extract JSON from text, handling reasoning tags and incomplete JSON."""
    if not text:
        return None
    
    # Remove reasoning tags (handle both formats)
    text = re.sub(r'`<think>`.*?`</think>`', '', text, flags=re.DOTALL)
    text = re.sub(r'`<think>`.*?`</think>`', '', text, flags=re.DOTALL)
    text = text.strip()
    
    # Find the first opening brace
    first_brace = text.find('{')
    if first_brace == -1:
        print("No opening brace found")
        return None
    
    # Start from the first brace
    json_candidate = text[first_brace:]
    
    # Try to find matching closing brace by counting braces
    brace_count = 0
    last_valid_pos = -1
    
    for i, char in enumerate(json_candidate):
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
            if brace_count == 0:
                last_valid_pos = i
                break
    
    if last_valid_pos == -1:
        print("⚠ JSON appears incomplete (no matching closing brace)")
        print(f"Brace count at end: {brace_count} (should be 0)")
        # Try to parse anyway and see the error
        try:
            # Try parsing what we have
            test_json = json_candidate[:5000]  # First 5000 chars
            json.loads(test_json)
        except json.JSONDecodeError as e:
            print(f"JSON error at position {e.pos}: {e.msg}")
            print(f"Context: ...{test_json[max(0, e.pos-50):e.pos+50]}...")
        return None
    
    # Extract the complete JSON
    json_str = json_candidate[:last_valid_pos + 1]
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"JSON decode error at position {e.pos}: {e.msg}")
        print(f"Error context: ...{json_str[max(0, e.pos-50):e.pos+50]}...")
        return None

# Test it
map_json = extract_json_from_text(generated_content)

if map_json:
    print("✓ Successfully extracted JSON!")
    print(f"Keys: {list(map_json.keys())}")
    
    # Validate it has the required fields
    required = ['tiles', 'player_spawn', 'stairs_spawn', 'width', 'height']
    missing = [key for key in required if key not in map_json]
    if missing:
        print(f"⚠ Missing required fields: {missing}")
    else:
        print("✓ All required fields present")
        
        map_id = "map_generated_001"
        with open(f"../web_game/maps/{map_id}.json", 'w') as f:
            json.dump(map_json, f, indent=2)
        print(f"✓ Saved to web_game/maps/{map_id}.json")
else:
    print("✗ Could not extract valid JSON")
    # Show where the JSON starts
    json_start = generated_content.find('{')
    if json_start != -1:
        print(f"\nJSON starts at position {json_start}")
        print(f"First 200 chars of JSON: {generated_content[json_start:json_start+200]}")
        print(f"Last 200 chars of generated content: {generated_content[-200:]}")

In [ ]:
index_file = "../web_game/maps/map_index.json"
with open(index_file, 'r') as f:
    index = json.load(f)

if map_id not in index["map_ids"]:
    index["map_ids"].append(map_id)
    index["total_maps"] = len(index["map_ids"])

    with open(index_file, 'w') as f:
        json.dump(index, f, indent=2)
    print(f"Added {map_id} to index")